# 05 — Score the 600 open orders

Run top to bottom in a **fresh authenticated lakehouse_to_ai AIDP session**, with
native Experiments/Models and Spark/Delta compute. Install project wheel
**aidp_lakehouse_to_ai-0.6.2-py3-none-any.whl** without replacing native dependencies,
then restart Python. No state from notebook 04 is required.

**User-reported evidence:** TRAINING PASS, model
`lakehouse_demo.gold.late_delivery_acceptance`, version **1**, run
`fc64485e353c41168cb90ee0f0bc5f71`; exact-version reload passed. Threshold 0.25,
cutoff 2026-04-01T00:00:00Z, zero open orders scored previously. These last two
values are recovered and cross-checked from the saved model/run, never configured
as separate scoring rules. **Live AIDP scoring remains pending.**

This notebook reads one pinned initial LF v2 Gold snapshot and writes only
`lakehouse_demo.gold.late_delivery_predictions`. It never fits a model, modifies
source layers, deploys an endpoint or publishes to Autonomous. Expect 600 unique
predictions, identical results on repeat and zero additional rows on the second
publication. Existing incompatible definitions or contents cause a stop.
See `docs/late-delivery-scoring.md` for permissions, provenance and recovery.

**Connection diagnosis:** run only configuration and the native metadata-binding
cell below, then stop. Do not Run All, load the model or execute scoring while
connection diagnosis is in progress. See docs/aidp-native-mlflow-binding.md.


In [ ]:
# The only configuration section: identity and source, never a duplicate decision rule.
CONFIG = {
    "source": {
        "catalog": "lakehouse_demo", "namespace": "frgpslcqn7sm",
        "bucket": "CristinaVaras-bucket-AIDP",
        "prefix": "demos/lakehouse-to-ai/source-data/v2/seed-42/reference-2026-10-28",
        "reference_date": "2026-10-28", "seed": 42, "batch_id": "initial",
    },
    "registered_model_name": "lakehouse_demo.gold.late_delivery_acceptance",
    "model_version": "1",
    "training_run_id": "fc64485e353c41168cb90ee0f0bc5f71",
    "native_backend": {
        "tracking_uri": "https://datalake.eu-frankfurt-1.oci.oraclecloud.com/20260430/aiDataPlatforms/ocid1.aidataplatform.oc1.eu-frankfurt-1.amaaaaaal7flieyam7ksfck72a3nedbu2zghvv3zuart6j5ipsflxsaifq5a/workspaces/82a342a2-c75d-462f-9b66-4f3a9abcfc49/mlops",
        "registry_uri": "https://datalake.eu-frankfurt-1.oci.oraclecloud.com/20260430/aiDataPlatforms/ocid1.aidataplatform.oc1.eu-frankfurt-1.amaaaaaal7flieyam7ksfck72a3nedbu2zghvv3zuart6j5ipsflxsaifq5a/mlops",
        "experiment_id": "1",
        "experiment_name": "lakehouse_to_ai_late_delivery"
    },
    "input_delta_version": None,  # Resolve once in section 2; both inferences reuse it.
}
cfg = native_connection = bundle = capabilities = data = snapshot = verification = None

## 1a. Native workspace binding — metadata reads only

These two non-secret MLOps bases were extracted from successful documented
ask-aidp GET responses for this exact instance/workspace, not from the UI URL.
The installed native mlops_auth_provider and Spark/Java signer are preserved.
An explicit client must read the existing experiment, run and version successfully
before any global routing is changed; the effective default client is then checked.
A rejected native principal leaves routing/auth unchanged. No model is loaded here.
**Stop after this cell during diagnosis**, even when NATIVE CONNECTION PASS appears.

In [ ]:
cfg = native_connection = bundle = capabilities = data = snapshot = verification = None
try:
    import json
    import aidp_lakehouse
    from aidp_lakehouse.aidp_bronze import BronzeConfig
    from aidp_lakehouse.gold import GoldConfig
    from aidp_lakehouse.scoring import ScoringConfig, VERSION
    from aidp_lakehouse.native_mlflow import NativeBackendConfig, initialize_native_backend
    if aidp_lakehouse.__version__ != VERSION:
        raise RuntimeError("Install the 0.6.2 wheel and restart Python")
    spark.version
    cfg = ScoringConfig(gold=GoldConfig(BronzeConfig(**CONFIG["source"])),
        **{k: v for k, v in CONFIG.items() if k not in ("source", "native_backend")})
    native_connection = initialize_native_backend(NativeBackendConfig(**CONFIG["native_backend"]), cfg)
    print(json.dumps(native_connection, indent=2))
    print("Connection diagnosis: STOP HERE. No model loaded; no open order scored.")
except BaseException:
    cfg = native_connection = bundle = capabilities = data = snapshot = verification = None
    print("NATIVE CONNECTION FAIL: do not run model loading or scoring cells")
    raise

## 1b. Runtime, native registry identity and saved contracts

Preflight performs read-only checks; it preserves native authentication and URIs.
Load only the exact configured version, verify its training run and FINISHED
status, and recover the feature contract, threshold and cutoff from its pipeline
and artifacts. Missing or conflicting metadata stops before inference.

In [ ]:
bundle = capabilities = data = snapshot = verification = None
try:
    if native_connection is None or native_connection.get("status") != "NATIVE CONNECTION PASS":
        raise RuntimeError("Complete native metadata binding before loading the model")
    import json
    import aidp_lakehouse
    from aidp_lakehouse.aidp_bronze import BronzeConfig
    from aidp_lakehouse.gold import GoldConfig
    from aidp_lakehouse.scoring import ScoringConfig, VERSION
    from aidp_lakehouse.scoring_mlflow import load_verified_pipeline
    from aidp_lakehouse.scoring_spark import read_scoring_input, score_publish_repeat
    if aidp_lakehouse.__version__ != VERSION:
        raise RuntimeError("Install the 0.6.2 wheel and restart Python")
    spark.version  # An attached AIDP Spark session is required.
    spark.conf.set("spark.sql.session.timeZone", "UTC")
    cfg = ScoringConfig(gold=GoldConfig(BronzeConfig(**CONFIG["source"])),
                        **{k: v for k, v in CONFIG.items() if k not in ("source", "native_backend")})
    bundle, capabilities = load_verified_pipeline(cfg, require_aidp=True)
    print(json.dumps({"package": aidp_lakehouse.__version__, "spark": spark.version,
        "model_uri": cfg.model_uri, "training_run": cfg.training_run_id,
        "recovered_threshold": bundle.threshold, "recovered_cutoff": bundle.cutoff,
        "feature_contract": bundle.feature_contract, "runtime": capabilities}, indent=2))
except BaseException:
    bundle = capabilities = data = snapshot = verification = None
    print("SCORING FAIL: native model/contract preflight did not complete")
    raise

## 2. Pin and validate the Gold input

When input_delta_version is None, resolve the current Delta version once and
record it below. To reproduce this run in a later session, configure that integer.
Both inferences below use exactly these collected 600 rows. Only explicit Gold
predictors go into the pipeline; identifiers and snapshot metadata stay outside.

In [ ]:
data = snapshot = verification = None
try:
    if bundle is None or capabilities is None:
        raise RuntimeError("Complete native model preflight first")
    data, snapshot = read_scoring_input(spark, cfg)
    print(json.dumps({"rows": len(data), "fixed_snapshot": snapshot,
                      "output_table": cfg.output_table}, indent=2))
except BaseException:
    data = snapshot = verification = None
    print("SCORING FAIL: fixed Gold input validation did not complete")
    raise

## 3. Infer, publish, infer again and reconcile

This executes the complete saved pipeline without fitting. It checks class
ordering, probabilities, saved threshold consistency, exact coverage and UTC
cutoff compatibility before any publication. Repeat compares every output column
and row; insert-only Delta publication refuses changed or extra existing rows.
Do not run concurrent writers to the predictions table.

In [ ]:
verification = None  # Re-execution immediately revokes any previous PASS.
try:
    if bundle is None or data is None or snapshot is None:
        raise RuntimeError("Complete model and fixed-input checks first")
    verification = score_publish_repeat(spark, data, bundle, cfg, snapshot)
    print(json.dumps(verification, indent=2))
except BaseException:
    verification = None
    print("SCORING FAIL: inference/publication/repetition did not complete")
    raise

## 4. Distribution, alerts and deterministic risk ranking

Risk is the saved model's estimated probability of late delivery, not commercial
priority or a causal explanation. Higher scores rank first; equal scores are
ordered by order_id. No outcome labels or accuracy estimates exist for this open
cohort. Decile bins are descriptive reporting, not an alternative decision rule.

In [ ]:
if verification is None or verification.get("status") != "PASS":
    raise RuntimeError("Complete both scoring executions before inspecting results")
from pyspark.sql import functions as F
from pyspark.sql.window import Window
predictions = spark.table(cfg.output_table)
predictions.groupBy("predicted_late").count().orderBy("predicted_late").show()
(predictions.withColumn("score_decile", F.least(F.floor(F.col("late_probability") * 10), F.lit(9)))
 .groupBy("score_decile").count().orderBy("score_decile").show())
(predictions.withColumn("risk_rank", F.row_number().over(
    Window.orderBy(F.desc("late_probability"), F.asc("order_id"))))
 .select("risk_rank", "order_id", "customer_id", "late_probability", "predicted_late",
         "decision_threshold", "model_version", "input_delta_version")
 .orderBy("risk_rank").show(20, truncate=False))

In [ ]:
if verification is None or verification.get("status") != "PASS" or not verification["full_row_repeat_equal"]:
    raise RuntimeError("SCORING FAIL: repeat verification is absent or failed")
if verification["repeat"]["inserted_rows"] != 0 or verification["scoring"]["rows"] != 600:
    raise RuntimeError("SCORING FAIL: coverage or idempotency mismatch")
print("SCORING PASS: 600 exact-version predictions; full-row repeat identical; zero additional rows")